# Conditional DDPM — Butterfly Image Generation

Implementação de um **Denoising Diffusion Probabilistic Model (DDPM)** condicional por classe,  
adaptado ao projecto de augmentação de imagens de borboletas.

Baseia-se no paper original:  
*Denoising Diffusion Probabilistic Models* — Ho et al. (2020) — https://arxiv.org/abs/2006.11239

---
### Diferenças face ao notebook de referência (MNIST/FashionMNIST)
| Aspecto | Notebook original | Este notebook |
|---------|-------------------|---------------|
| Dataset | MNIST / FashionMNIST (28×28, 1 canal) | ButterflyDataset (64×64, 3 canais RGB) |
| Condicionamento | Sem condicionamento de classe | Condicional: class embedding somado ao time embedding |
| UNet | Arquitectura 28×28 fixa | UNet redimensionada para 64×64, 3 canais |
| Saída | Geração livre | Geração guiada por `GENERATE_COUNT_MAP` (config.py) |

## 0. Instalações


In [ ]:
!pip install einops --quiet
!pip install imageio[ffmpeg] --quiet

## 1. Imports e Config


In [ ]:
import os
import sys
import random
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.utils.data as data
import torchvision.transforms as transforms
from torch.optim import Adam
from torchvision.utils import save_image

import matplotlib.pyplot as plt
import einops
import imageio
from tqdm.auto import tqdm

sys.path.append(os.path.abspath('..'))
from src.config import TARGET_CLASSES, IMAGE_SIZE, BATCH_SIZE, GENERATE_COUNT_MAP
from src.utils.data_loader import ButterflyDataset
from src.utils.metrics import inception_score, frechet_inception_distance, mean_ssim

NUM_CLASSES = len(TARGET_CLASSES)
print(f"Classes       : {TARGET_CLASSES}")
print(f"NUM_CLASSES   : {NUM_CLASSES}")
print(f"IMAGE_SIZE    : {IMAGE_SIZE}")
print(f"BATCH_SIZE    : {BATCH_SIZE}")
print(f"Count map     : {GENERATE_COUNT_MAP}")

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = (
    torch.device('cuda')  if torch.cuda.is_available()        else
    torch.device('mps')   if torch.backends.mps.is_available() else
    torch.device('cpu')
)
print('Device:', device)

## 2. Dataset — ButterflyDataset

Normalização para `[-1, 1]` tal como no DDPM original (alinha com a distribuição Gaussiana do ruído).


In [ ]:
path    = 'splits'
img_dir = '../data/raw/train'
df      = pd.read_csv(os.path.join(path, 'full_train_split.csv'))

data_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5],   # [0,1] → [-1, 1]
                         [0.5, 0.5, 0.5]),
])

dataset    = ButterflyDataset(df=df, img_dir=img_dir,
                               transform=data_transform,
                               target_classes=None)
dataloader = data.DataLoader(dataset, batch_size=BATCH_SIZE,
                              shuffle=True, num_workers=2, pin_memory=True)
print(f'Dataset : {len(dataset)} imagens | {len(dataloader)} batches/época')

## 3. Funções Utilitárias


In [ ]:
def show_images(images, title=''):
    """Mostra imagens em grelha quadrada. Suporta tensores RGB ou grayscale."""
    if isinstance(images, torch.Tensor):
        images = images.detach().cpu()
        # Desnormalizar de [-1,1] para [0,1]
        images = (images + 1) * 0.5
        images = images.clamp(0, 1).numpy()
    n   = len(images)
    rows = int(n ** 0.5)
    cols = (n + rows - 1) // rows
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 2, rows * 2))
    axes = np.array(axes).flatten()
    for i, ax in enumerate(axes):
        if i < n:
            img = np.transpose(images[i], (1, 2, 0))   # C,H,W → H,W,C
            ax.imshow(img.clip(0, 1))
        ax.axis('off')
    fig.suptitle(title, fontsize=14)
    plt.tight_layout()
    plt.show()


def show_first_batch(loader):
    for imgs, labels in loader:
        show_images(imgs[:16], 'Primeiras imagens do dataset (desnormalizadas)')
        print('Labels:', labels[:16].tolist())
        break

show_first_batch(dataloader)

## 4. Módulo DDPM

O `MyDDPM` encapsula o schedule de ruído (betas lineares) e expõe:
- **`forward(x0, t, eta)`** — processo de adição de ruído (skip directo ao passo $t$)
- **`backward(x, t)`** — delega à rede UNet para prever o ruído $\epsilon_\theta(x_t, t, c)$


In [ ]:
class MyDDPM(nn.Module):
    """DDPM wrapper — independente da arquitectura da rede."""
    def __init__(self, network, n_steps=1000,
                 min_beta=1e-4, max_beta=0.02,
                 device=None, image_chw=(3, 64, 64)):
        super().__init__()
        self.n_steps   = n_steps
        self.device    = device
        self.image_chw = image_chw
        self.network   = network.to(device)
        self.betas     = torch.linspace(min_beta, max_beta, n_steps).to(device)
        self.alphas    = 1 - self.betas
        self.alpha_bars = torch.tensor(
            [torch.prod(self.alphas[:i + 1]) for i in range(len(self.alphas))]
        ).to(device)

    def forward(self, x0, t, eta=None):
        """Processo forward: adiciona ruído a x0 no passo t."""
        n, c, h, w = x0.shape
        a_bar = self.alpha_bars[t]
        if eta is None:
            eta = torch.randn(n, c, h, w).to(self.device)
        noisy = (a_bar.sqrt().reshape(n, 1, 1, 1) * x0
                 + (1 - a_bar).sqrt().reshape(n, 1, 1, 1) * eta)
        return noisy

    def backward(self, x, t, labels=None):
        """Processo backward: estima o ruído adicionado. Passes labels para condicionamento."""
        return self.network(x, t, labels)

## 5. UNet Condicional para 64×64 RGB

Adaptações relativamente à UNet original (28×28, 1 canal):
- Entrada: **3 canais RGB** (em vez de 1)
- Dimensões espaciais: **64×64** → hierarquia de resolução ajustada
- **Class embedding** (`nn.Embedding`) somado ao time embedding em cada bloco,  
  permitindo que o modelo condicione a geração à espécie de borboleta


In [ ]:
def sinusoidal_embedding(n, d):
    """Positional/time embedding sinusoidal (idêntico ao paper)."""
    emb = torch.zeros(n, d)
    wk  = torch.tensor([1 / 10_000 ** (2 * j / d) for j in range(d)]).reshape(1, d)
    t   = torch.arange(n).reshape(n, 1)
    emb[:, ::2]  = torch.sin(t * wk[:, ::2])
    emb[:, 1::2] = torch.cos(t * wk[:, ::2])
    return emb


class ResBlock(nn.Module):
    """Bloco residual com LayerNorm + 2×Conv2d + SiLU."""
    def __init__(self, shape, in_c, out_c,
                 kernel_size=3, stride=1, padding=1,
                 normalize=True):
        super().__init__()
        self.ln       = nn.LayerNorm(shape)
        self.conv1    = nn.Conv2d(in_c,  out_c, kernel_size, stride, padding)
        self.conv2    = nn.Conv2d(out_c, out_c, kernel_size, stride, padding)
        self.act      = nn.SiLU()
        self.normalize = normalize
        # Projecção residual se os canais forem diferentes
        self.skip = nn.Conv2d(in_c, out_c, 1) if in_c != out_c else nn.Identity()

    def forward(self, x):
        out = self.ln(x) if self.normalize else x
        out = self.act(self.conv1(out))
        out = self.act(self.conv2(out))
        return out + self.skip(x)


class ConditionalUNet(nn.Module):
    """
    UNet condicional para imagens 64×64 RGB.

    Condicionamento dual:
      - Time embedding  : sinusoidal  → MLP → adicionado a cada nível
      - Class embedding : nn.Embedding → adicionado ao time embedding

    Arquitectura:
      Encoder: 64 → 32 → 16 → 8
      Bottleneck: 4
      Decoder:  8 → 16 → 32 → 64  (skip connections)
    """
    def __init__(self, n_steps=1000, time_emb_dim=256,
                 num_classes=NUM_CLASSES, base_ch=64, nc=3):
        super().__init__()
        self.nc = nc

        # ── Time embedding (sinusoidal + learnable) ──────────────────────
        self.time_embed = nn.Embedding(n_steps, time_emb_dim)
        self.time_embed.weight.data = sinusoidal_embedding(n_steps, time_emb_dim)
        self.time_embed.requires_grad_(False)

        # ── Class embedding ───────────────────────────────────────────────
        self.class_embed = nn.Embedding(num_classes, time_emb_dim)

        # helper: time+class → channel
        def te(dim_out):
            return nn.Sequential(
                nn.Linear(time_emb_dim, dim_out),
                nn.SiLU(),
                nn.Linear(dim_out, dim_out)
            )

        ch = base_ch   # 64

        # ── Encoder ──────────────────────────────────────────────────────
        # Level 1: 64×64, nc → ch
        self.te1 = te(nc)
        self.b1  = nn.Sequential(
            ResBlock((nc, 64, 64), nc, ch),
            ResBlock((ch, 64, 64), ch, ch),
        )
        self.down1 = nn.Conv2d(ch, ch, 4, 2, 1)   # 64→32

        # Level 2: 32×32, ch → 2ch
        self.te2 = te(ch)
        self.b2  = nn.Sequential(
            ResBlock((ch, 32, 32), ch, ch*2),
            ResBlock((ch*2, 32, 32), ch*2, ch*2),
        )
        self.down2 = nn.Conv2d(ch*2, ch*2, 4, 2, 1)  # 32→16

        # Level 3: 16×16, 2ch → 4ch
        self.te3 = te(ch*2)
        self.b3  = nn.Sequential(
            ResBlock((ch*2, 16, 16), ch*2, ch*4),
            ResBlock((ch*4, 16, 16), ch*4, ch*4),
        )
        self.down3 = nn.Conv2d(ch*4, ch*4, 4, 2, 1)  # 16→8

        # Level 4: 8×8, 4ch → 8ch
        self.te4 = te(ch*4)
        self.b4  = nn.Sequential(
            ResBlock((ch*4, 8, 8), ch*4, ch*8),
            ResBlock((ch*8, 8, 8), ch*8, ch*8),
        )
        self.down4 = nn.Conv2d(ch*8, ch*8, 4, 2, 1)  # 8→4

        # ── Bottleneck ────────────────────────────────────────────────────
        self.te_mid = te(ch*8)
        self.b_mid  = nn.Sequential(
            ResBlock((ch*8, 4, 4), ch*8, ch*8),
            ResBlock((ch*8, 4, 4), ch*8, ch*8),
        )

        # ── Decoder ───────────────────────────────────────────────────────
        self.up4   = nn.ConvTranspose2d(ch*8,  ch*8,  4, 2, 1)   # 4→8
        self.te_d4 = te(ch*16)
        self.b_d4  = nn.Sequential(
            ResBlock((ch*16, 8, 8),  ch*16, ch*4),
            ResBlock((ch*4,  8, 8),  ch*4,  ch*4),
        )

        self.up3   = nn.ConvTranspose2d(ch*4,  ch*4,  4, 2, 1)   # 8→16
        self.te_d3 = te(ch*8)
        self.b_d3  = nn.Sequential(
            ResBlock((ch*8,  16, 16), ch*8,  ch*2),
            ResBlock((ch*2,  16, 16), ch*2,  ch*2),
        )

        self.up2   = nn.ConvTranspose2d(ch*2,  ch*2,  4, 2, 1)   # 16→32
        self.te_d2 = te(ch*4)
        self.b_d2  = nn.Sequential(
            ResBlock((ch*4,  32, 32), ch*4,  ch),
            ResBlock((ch,    32, 32), ch,    ch),
        )

        self.up1   = nn.ConvTranspose2d(ch,    ch,    4, 2, 1)   # 32→64
        self.te_d1 = te(ch*2)
        self.b_d1  = nn.Sequential(
            ResBlock((ch*2,  64, 64), ch*2,  ch),
            ResBlock((ch,    64, 64), ch,    ch, normalize=False),
        )

        self.conv_out = nn.Conv2d(ch, nc, 3, 1, 1)   # → 3 canais RGB

    # ── forward ──────────────────────────────────────────────────────────────
    def forward(self, x, t, labels=None):
        """
        x      : (N, 3, 64, 64)  imagem ruidosa
        t      : (N, 1)          time-step (long)
        labels : (N,)            índice de classe (long); None → treino livre
        """
        n = len(x)
        # Time embedding + class embedding (somados)
        t_emb = self.time_embed(t.squeeze(-1))       # (N, time_emb_dim)
        if labels is not None:
            t_emb = t_emb + self.class_embed(labels) # condicionamento de classe

        def add_te(te_module, tensor):
            return tensor + te_module(t_emb).reshape(n, -1, 1, 1)

        # Encoder
        e1 = self.b1(add_te(self.te1, x))                      # (N, ch,   64, 64)
        e2 = self.b2(add_te(self.te2, self.down1(e1)))          # (N, 2ch,  32, 32)
        e3 = self.b3(add_te(self.te3, self.down2(e2)))          # (N, 4ch,  16, 16)
        e4 = self.b4(add_te(self.te4, self.down3(e3)))          # (N, 8ch,   8,  8)

        # Bottleneck
        mid = self.b_mid(add_te(self.te_mid, self.down4(e4)))   # (N, 8ch,   4,  4)

        # Decoder (skip connections)
        d4 = self.b_d4(add_te(self.te_d4,
                               torch.cat([e4, self.up4(mid)], dim=1)))   # (N, 4ch, 8, 8)
        d3 = self.b_d3(add_te(self.te_d3,
                               torch.cat([e3, self.up3(d4)],  dim=1)))   # (N, 2ch,16,16)
        d2 = self.b_d2(add_te(self.te_d2,
                               torch.cat([e2, self.up2(d3)],  dim=1)))   # (N,  ch,32,32)
        d1 = self.b_d1(add_te(self.te_d1,
                               torch.cat([e1, self.up1(d2)],  dim=1)))   # (N,  ch,64,64)

        return self.conv_out(d1)   # (N, 3, 64, 64)

## 6. Instanciar o Modelo


In [ ]:
# Hyperparâmetros (seguindo o paper original)
N_STEPS  = 1000
MIN_BETA = 1e-4
MAX_BETA = 0.02

unet = ConditionalUNet(
    n_steps      = N_STEPS,
    time_emb_dim = 256,
    num_classes  = NUM_CLASSES,
    base_ch      = 64,
    nc           = 3
)

ddpm = MyDDPM(
    network    = unet,
    n_steps    = N_STEPS,
    min_beta   = MIN_BETA,
    max_beta   = MAX_BETA,
    device     = device,
    image_chw  = (3, IMAGE_SIZE, IMAGE_SIZE)
)

n_params = sum(p.numel() for p in ddpm.parameters())
print(f'Parâmetros totais: {n_params:,}')

## 7. Visualização do Processo Forward

Verifica que o ruído é adicionado correctamente nas borboletas.


In [ ]:
def show_forward(ddpm, loader, device, n_show=8):
    for imgs, labels in loader:
        imgs = imgs[:n_show].to(device)
        show_images(imgs, 'Imagens originais')
        for pct in [0.25, 0.5, 0.75, 1.0]:
            t_val = int(pct * ddpm.n_steps) - 1
            noisy = ddpm(imgs, [t_val] * n_show)
            show_images(noisy, f'Ruído {int(pct*100)}% (t={t_val})')
        break

show_forward(ddpm, dataloader, device)

## 8. Geração de Novas Imagens (Backward / Sampling)

Duas opções de $\sigma_t^2$ (seguindo o paper Ho et al.):
- **Option 1**: $\sigma_t^2 = \beta_t$
- **Option 2**: $\sigma_t^2 = \tilde{\beta}_t = \frac{1-\bar{\alpha}_{t-1}}{1-\bar{\alpha}_t}\beta_t$

O argumento `labels` permite geração condicional por classe de borboleta.


In [ ]:
def generate_new_images(
        ddpm, n_samples=16, option=1, device=None,
        labels=None,
        frames_per_gif=100, gif_name='sampling.gif',
        c=3, h=64, w=64):
    """
    Gera n_samples imagens a partir de ruído Gaussiano.
    labels : tensor (n_samples,) com índices de classe; None = livre
    """
    frame_idxs = np.linspace(0, ddpm.n_steps, frames_per_gif).astype(np.uint32)
    frames = []

    with torch.no_grad():
        if device is None:
            device = ddpm.device

        x = torch.randn(n_samples, c, h, w).to(device)

        for idx, t in enumerate(list(range(ddpm.n_steps))[::-1]):
            time_tensor = (torch.ones(n_samples, 1) * t).to(device).long()
            lbl_tensor  = labels.to(device) if labels is not None else None

            eta_theta = ddpm.backward(x, time_tensor, lbl_tensor)

            alpha_t     = ddpm.alphas[t]
            alpha_t_bar = ddpm.alpha_bars[t]

            x = (1 / alpha_t.sqrt()) * (
                x - (1 - alpha_t) / (1 - alpha_t_bar).sqrt() * eta_theta
            )

            if t > 0:
                z      = torch.randn(n_samples, c, h, w).to(device)
                beta_t = ddpm.betas[t]
                if option == 1:
                    sigma_t = beta_t.sqrt()
                else:
                    prev_ab = ddpm.alpha_bars[t - 1] if t > 0 else ddpm.alphas[0]
                    sigma_t = (((1 - prev_ab) / (1 - alpha_t_bar)) * beta_t).sqrt()
                x = x + sigma_t * z

            # GIF frame
            if idx in frame_idxs or t == 0:
                norm = x.clone()
                for i in range(len(norm)):
                    norm[i] -= norm[i].min()
                    norm[i] *= 255 / (norm[i].max() + 1e-8)
                frame = einops.rearrange(
                    norm, '(b1 b2) c h w -> (b1 h) (b2 w) c',
                    b1=int(n_samples ** 0.5)
                )
                frames.append(frame.cpu().numpy().astype(np.uint8))

    with imageio.get_writer(gif_name, mode='I') as writer:
        for i, frame in enumerate(frames):
            writer.append_data(frame)
            if i == len(frames) - 1:
                for _ in range(frames_per_gif // 3):
                    writer.append_data(frame)

    return x

## 9. Training Loop

Loss: **MSE** entre o ruído real $\epsilon$ e a previsão da rede $\epsilon_\theta(x_t, t, c)$.

$$\mathcal{L} = \mathbb{E}_{t,x_0,\epsilon}\left[\|\epsilon - \epsilon_\theta(\sqrt{\bar{\alpha}_t}x_0 + \sqrt{1-\bar{\alpha}_t}\epsilon,\; t,\; c)\|^2\right]$$


In [ ]:
def training_loop(
        ddpm, loader, n_epochs, optim, device,
        display=False,
        store_path='checkpoints/ddpm_butterfly.pt'):

    os.makedirs('checkpoints', exist_ok=True)
    mse       = nn.MSELoss()
    best_loss = float('inf')
    n_steps   = ddpm.n_steps

    for epoch in tqdm(range(n_epochs), desc='Training DDPM', colour='#00ff00'):
        epoch_loss = 0.0
        ddpm.network.train()

        for step, (x0, labels) in enumerate(
                tqdm(loader, leave=False,
                     desc=f'Época {epoch+1}/{n_epochs}', colour='#005500')):

            x0     = x0.to(device)
            labels = labels.to(device)   # índice de classe
            n      = len(x0)

            # 1. Ruído aleatório e timestep por imagem
            eta = torch.randn_like(x0).to(device)
            t   = torch.randint(0, n_steps, (n,)).to(device)

            # 2. Forward: adiciona ruído
            noisy_imgs = ddpm(x0, t, eta)

            # 3. Backward: previsão do ruído (com condicionamento de classe)
            eta_theta = ddpm.backward(noisy_imgs, t.reshape(n, -1), labels)

            # 4. Optimização MSE
            loss = mse(eta, eta_theta)
            optim.zero_grad()
            loss.backward()
            optim.step()

            epoch_loss += loss.item() * n / len(loader.dataset)

        log = f'Época {epoch+1}/{n_epochs}  Loss: {epoch_loss:.4f}'

        if best_loss > epoch_loss:
            best_loss = epoch_loss
            torch.save(ddpm.state_dict(), store_path)
            log += '  ← melhor modelo guardado'

        print(log)

        if display and (epoch + 1) % 5 == 0:
            ddpm.network.eval()
            sample_lbl = torch.arange(NUM_CLASSES, device=device).repeat(
                (16 // NUM_CLASSES) + 1)[:16]
            gen = generate_new_images(ddpm, n_samples=16, device=device,
                                      labels=sample_lbl, gif_name='/dev/null')
            show_images(gen, f'Geradas — época {epoch+1}')

## 10. Treinar o Modelo


In [ ]:
N_EPOCHS  = 200
LR        = 1e-4
NO_TRAIN  = False   # True → salta treino, carrega checkpoint
STORE_PATH = 'checkpoints/ddpm_butterfly.pt'

if not NO_TRAIN:
    optimizer = Adam(ddpm.parameters(), lr=LR)
    training_loop(
        ddpm, dataloader,
        n_epochs   = N_EPOCHS,
        optim      = optimizer,
        device     = device,
        display    = True,
        store_path = STORE_PATH
    )
else:
    print('Treino ignorado — a carregar checkpoint...')

## 11. Carregar o Melhor Modelo


In [ ]:
best_unet = ConditionalUNet(
    n_steps=N_STEPS, time_emb_dim=256,
    num_classes=NUM_CLASSES, base_ch=64, nc=3
)
best_ddpm = MyDDPM(best_unet, n_steps=N_STEPS,
                   min_beta=MIN_BETA, max_beta=MAX_BETA, device=device)
best_ddpm.load_state_dict(torch.load(STORE_PATH, map_location=device))
best_ddpm.eval()
print('Modelo carregado com sucesso.')

## 12. Geração Condicional por Classe

Gera 4 amostras por classe (5 × 4 = 20 imagens) para inspecção visual.


In [ ]:
N_PER_CLASS = 4
sample_labels = torch.arange(NUM_CLASSES, device=device).repeat_interleave(N_PER_CLASS)
n_samples_vis  = NUM_CLASSES * N_PER_CLASS

print('Gerando amostras condicionais (Option 1)...')
gen_opt1 = generate_new_images(
    best_ddpm,
    n_samples       = n_samples_vis,
    option          = 1,
    device          = device,
    labels          = sample_labels,
    frames_per_gif  = 100,
    gif_name        = 'butterfly_opt1.gif',
    c=3, h=IMAGE_SIZE, w=IMAGE_SIZE
)
show_images(gen_opt1, 'Option 1 — σ²=β_t')

print('\nGerando amostras condicionais (Option 2)...')
gen_opt2 = generate_new_images(
    best_ddpm,
    n_samples       = n_samples_vis,
    option          = 2,
    device          = device,
    labels          = sample_labels,
    frames_per_gif  = 100,
    gif_name        = 'butterfly_opt2.gif',
    c=3, h=IMAGE_SIZE, w=IMAGE_SIZE
)
show_images(gen_opt2, 'Option 2 — σ²=β̃_t')

## 13. Visualizar o GIF de Difusão


In [ ]:
from IPython.display import Image as IPImage
IPImage(open('butterfly_opt1.gif', 'rb').read())

## 14. Guardar Imagens Geradas (`data/augmented`)

Usa o `GENERATE_COUNT_MAP` do `config.py` para gerar exactamente as imagens  
necessárias para cada classe, guardando-as em `../data/augmented/<class_name>/`.


In [ ]:
OUTPUT_DIR = '../data/augmented'

def generate_and_save_ddpm(
        ddpm, generate_count_map, device,
        target_classes, output_dir=OUTPUT_DIR,
        model_name='ddpm', option=1):
    """
    Gera imagens condicionais por classe com o DDPM e guarda em
    output_dir/<class_name>/.

    Args:
        ddpm              : modelo MyDDPM treinado
        generate_count_map: dict {class_idx: n_images}  (de config.py)
        device            : torch.device
        target_classes    : lista de nomes de classe
        output_dir        : directório raiz de saída
        model_name        : prefixo do ficheiro
        option            : 1 ou 2 (opção σ_t)
    Returns:
        list of tensors (C,H,W) em [0,1]
    """
    ddpm.eval()
    all_generated = []
    c, h, w = ddpm.image_chw

    for class_idx, num_to_gen in generate_count_map.items():
        class_name = target_classes[class_idx]
        save_dir   = os.path.join(output_dir, class_name)
        os.makedirs(save_dir, exist_ok=True)

        labels = torch.full((num_to_gen,), class_idx,
                             dtype=torch.long, device=device)

        imgs = generate_new_images(
            ddpm, n_samples=num_to_gen, option=option,
            device=device, labels=labels,
            frames_per_gif=10, gif_name='/dev/null',
            c=c, h=h, w=w
        )   # tensor (num_to_gen, 3, H, W) em [-1, 1]

        imgs01 = ((imgs.cpu() + 1) * 0.5).clamp(0, 1)

        for j, img in enumerate(imgs01):
            fname = os.path.join(save_dir, f'{model_name}_{j:04d}.png')
            save_image(img, fname)

        print(f'  [{class_name}] {num_to_gen} imagens guardadas em {save_dir}')
        all_generated.append(imgs01)

    return list(torch.cat(all_generated, dim=0))


total_expected = sum(GENERATE_COUNT_MAP.values())
print(f'Total de imagens a gerar (config): {total_expected}')
print(f'Destino: {OUTPUT_DIR}/<class_name>/')

In [ ]:
print('Gerando com DDPM (Option 1)...')
fake_imgs_ddpm = generate_and_save_ddpm(
    best_ddpm,
    generate_count_map = GENERATE_COUNT_MAP,
    device             = device,
    target_classes     = TARGET_CLASSES,
    model_name         = 'ddpm',
    option             = 1
)
print(f'\nTotal gerado: {len(fake_imgs_ddpm)} imagens')

## 15. Métricas de Avaliação

| Métrica | Significado | Melhor |
|---------|-------------|--------|
| **IS**   | Qualidade + diversidade das imagens geradas | ↑ maior |
| **FID**  | Distância entre distribuição real e gerada  | ↓ menor |
| **SSIM** | Similaridade estrutural pixel-a-pixel       | ↑ maior |


In [ ]:
# Recolher imagens reais para FID / SSIM
print('Recolhendo imagens reais...')
real_list = []
with torch.no_grad():
    for imgs, _ in dataloader:
        real_list.append((imgs + 1) * 0.5)   # [-1,1] → [0,1]
        if sum(x.size(0) for x in real_list) >= total_expected:
            break
real_imgs_all = list(torch.cat(real_list, dim=0)[:total_expected])
print(f'Imagens reais recolhidas: {len(real_imgs_all)}')

In [ ]:
metrics_device = 'cuda' if torch.cuda.is_available() else 'cpu'

print('=' * 55)
print('       AVALIAÇÃO — DDPM CONDICIONAL')
print('=' * 55)

is_mean, is_std = inception_score(
    fake_imgs_ddpm, batch_size=32, splits=5, device=metrics_device
)
fid_val  = frechet_inception_distance(
    real_imgs_all, fake_imgs_ddpm, batch_size=32, device=metrics_device
)
ssim_val = mean_ssim(real_imgs_all, fake_imgs_ddpm)

print(f'  IS   (↑ melhor): {is_mean:.4f} ± {is_std:.4f}')
print(f'  FID  (↓ melhor): {fid_val:.4f}')
print(f'  SSIM (↑ melhor): {ssim_val:.4f}')
print('=' * 55)